## 1. Định Nghĩa Hệ Thống & Bài Toán (System and Problem Definition)

**Hệ thống thông minh:** Hệ thống hỗ trợ định giá bất động sản tự động tại các đô thị lớn ở Việt Nam.

**Phát biểu bài toán hình thức:** Cho trước vector đặc trưng $\mathbf{x} \in \mathbb{R}^d$ biểu diễn thông số vật lý và pháp lý của căn nhà, tìm hàm $f_\theta(\mathbf{x}): \mathbb{R}^d \rightarrow \mathbb{R}^+$ nhằm cực tiểu hóa sai số toàn phương trung bình $\text{MSE}$ để ước tính chính xác giá trị thị trường $y$ (Tỷ VNĐ) của căn nhà.

## 2. Sơ Đồ Hệ Thống Thông Minh (Intelligent System Diagram)

```
[ Môi trường: Thị trường BĐS ] ---> [ Nhận thức: 11 thông số nhà ] ---> [ Biểu diễn: Vector x in R^d ]
                                                                                |
                                                                                v
[ Ứng dụng Web/Mobile ] <--- [ Quyết định: Giá nhà Tỷ VNĐ & Đơn giá ] <--- [ Mô hình: Random Forest ]
                                                ^
                                                |
                           [ Đồ thị Tri thức Batdongsan.com.vn ]
```

In [ ]:
print("Pipeline Định giá BĐS: Vietnam Housing Data -> Encoding -> Machine Learning -> Price Prediction -> Streamlit UI")


## 3. Nguồn Dữ Liệu Thực Tế (Dataset Source)

• **Tên tập dữ liệu:** Vietnam Housing Dataset 2024
• **Nguồn phát hành:** KaggleHub / Batdongsan Crawled Data
• **URL:** https://www.kaggle.com/datasets/nguyentiennhan/vietnam-housing-dataset-2024
• **Quy mô:** Hơn 22,000 bản ghi, dung lượng ~3.5 MB.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

data_path = os.path.join('data', 'vietnam_housing_dataset.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('..', '..', 'DATA', 'gianha.csv')
df = pd.read_csv(data_path)
print("Kích thước dữ liệu gốc:", df.shape)
df.head(3)


## 4. Mô Tả Tập Dữ Liệu (Dataset Description)

**Giải đáp 10 câu hỏi dữ liệu bắt buộc:**
1. *Hiện tượng thực tế:* Giao dịch và định giá bất động sản nhà ở tại Việt Nam.
2. *Một quan sát:* Một tin đăng mua bán căn nhà cụ thể.
3. *Các đặc trưng:* Area, Frontage, Access Road, Floors, Bedrooms, Bathrooms, Legal, Direction, City...
4. *Biến mục tiêu:* `Price` (Giá nhà tính bằng Tỷ VNĐ).
5. *Kiểu biến mục tiêu:* Số thực liên tục (Continuous Numerical).
6. *Loại bài toán:* Học có giám sát - Hồi quy (Regression).
7. *Số quan sát:* > 22,000 dòng.
8. *Số đặc trưng:* 11 đặc trưng.
9. *Đặc trưng số học:* Area, Frontage, Access Road, Floors, Bedrooms, Bathrooms.
10. *Đặc trưng phân loại:* Legal status, Direction, City, Furniture.

In [ ]:
print("Thông tin kiểu dữ liệu và giá trị thiếu:")
df.info()


## 5. Biểu Diễn Dữ Liệu (Data Representation - The Central Idea)

Mã hóa các biến danh mục bằng `LabelEncoder`, loại bỏ nhiễu ngoại lai (Outliers) về giá và diện tích.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Tiền xử lý và làm sạch dữ liệu
df_clean = df.dropna(subset=['Price', 'Area']).copy()
df_clean = df_clean[(df_clean['Price'] > 0.3) & (df_clean['Price'] < 100)]
df_clean = df_clean[(df_clean['Area'] > 10) & (df_clean['Area'] < 500)]

# Chuẩn hóa tên cột
col_mapping = {
    'Access Road': 'Access_Road',
    'House direction': 'House_direction',
    'Balcony direction': 'Balcony_direction',
    'Legal status': 'Legal',
    'Furniture state': 'Furniture'
}
df_clean = df_clean.rename(columns=col_mapping)

# Trích xuất City từ Address nếu có
if 'Address' in df_clean.columns:
    df_clean['City'] = df_clean['Address'].apply(lambda x: x.split(',')[-1].strip() if isinstance(x, str) else 'Hà Nội')
    df_clean = df_clean.drop(columns=['Address'])

cat_cols = ['City', 'Legal', 'House_direction', 'Balcony_direction', 'Furniture']
encoders = {}
for col in cat_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('Unknown').astype(str)
        le = LabelEncoder()
        df_clean[col] = le.fit_transform(df_clean[col])
        encoders[col] = le

num_cols = ['Area', 'Frontage', 'Access_Road', 'Floors', 'Bedrooms', 'Bathrooms']
for col in num_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Dữ liệu sau khi mã hóa đặc trưng:")
df_clean.head(3)


## 6. Phân Tích Đặc Trưng & Mục Tiêu (Feature and Target Analysis)

Phân tích tương quan giữa các đặc trưng vật lý với giá bán căn nhà.

In [ ]:
corr_house = df_clean.corr(numeric_only=True)
print("Tương quan Pearson với giá nhà (Price):")
print(corr_house['Price'].sort_values(ascending=False))


## 7. Phân Tích Dữ Liệu Khám Phá (Exploratory Data Analysis - EDA)

Trực quan hóa phân phối giá nhà, diện tích và ma trận tương quan bất động sản.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_clean['Price'], bins=40, kde=True, color='#27ae60', ax=axes[0])
axes[0].set_title('1. Phân Phối Giá Nhà (Tỷ VNĐ)', fontweight='bold')

sample_size = min(1000, len(df_clean))
sns.scatterplot(data=df_clean.sample(sample_size, random_state=42), x='Area', y='Price', color='#2980b9', alpha=0.6, ax=axes[1])
axes[1].set_title('2. Quan Hệ Giữa Diện Tích (m²) & Giá Nhà', fontweight='bold')

sns.heatmap(corr_house[['Price']].sort_values(by='Price', ascending=False), annot=True, cmap='Greens', ax=axes[2])
axes[2].set_title('3. Hệ Số Tương Quan Với Giá Nhà', fontweight='bold')

plt.tight_layout()
plt.show()


## 8. Phân Chia Train / Test (Train/Test Split)

Phân chia 80% Training và 20% Testing với `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df_clean.columns if c != 'Price']
X = df_clean[feature_cols]
y = df_clean['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Kích thước tập Train: {X_train.shape}, Tập Test: {X_test.shape}")


## 9. Đường Cơ Sở Tham Chiếu (Baseline Reference Point)

Sử dụng `DummyRegressor(strategy='median')` làm mốc so sánh cơ sở.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

dummy = DummyRegressor(strategy='median')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

mae_base = mean_absolute_error(y_test, y_pred_dummy)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_dummy))
r2_base = r2_score(y_test, y_pred_dummy)

print(f"Baseline (Median) -> MAE: {mae_base:.2f} Tỷ, RMSE: {rmse_base:.2f} Tỷ, R2: {r2_base:.3f}")


## 10. Mô Hình 1: Linear Regression

Mô hình hồi quy tuyến tính cổ điển tối ưu hóa phần dư bình phương tối thiểu (OLS).

In [ ]:
from sklearn.linear_model import LinearRegression

m1 = LinearRegression()
m1.fit(X_train, y_train)
y_pred_m1 = m1.predict(X_test)

print(f"Linear Regression -> MAE: {mean_absolute_error(y_test, y_pred_m1):.2f} Tỷ, R2: {r2_score(y_test, y_pred_m1):.3f}")


## 11. Mô Hình 2: Decision Tree Regressor

Cây quyết định hồi quy phân chia không gian đặc trưng tối thiểu hóa MSE cục bộ.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

m2 = DecisionTreeRegressor(max_depth=8, random_state=42)
m2.fit(X_train, y_train)
y_pred_m2 = m2.predict(X_test)

print(f"Decision Tree (max_depth=8) -> MAE: {mean_absolute_error(y_test, y_pred_m2):.2f} Tỷ, R2: {r2_score(y_test, y_pred_m2):.3f}")


## 12. Mô Hình 3: HistGradientBoosting Regressor

Thuật toán Gradient Boosting tốc độ cao dựa trên histogram hóa đặc trưng.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

m3 = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.08, random_state=42)
m3.fit(X_train, y_train)
y_pred_m3 = m3.predict(X_test)

print(f"HistGradientBoosting -> MAE: {mean_absolute_error(y_test, y_pred_m3):.2f} Tỷ, R2: {r2_score(y_test, y_pred_m3):.3f}")


## 13. Mô Hình 4: Random Forest Regressor

Rừng ngẫu nhiên hồi quy kết hợp 200 cây quyết định độc lập giảm mạnh phương sai.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

m4 = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
m4.fit(X_train, y_train)
y_pred_m4 = m4.predict(X_test)

print(f"Random Forest Regressor -> MAE: {mean_absolute_error(y_test, y_pred_m4):.2f} Tỷ, R2: {r2_score(y_test, y_pred_m4):.3f}")


## 14. Đánh Giá Toàn Diện (Comprehensive Evaluation)

Bảng so sánh MAE, RMSE và $R^2$ Score giữa các thuật toán hồi quy.

In [ ]:
models_house = {
    'Baseline (Median)': y_pred_dummy,
    'Linear Regression': y_pred_m1,
    'Decision Tree': y_pred_m2,
    'HistGradientBoosting': y_pred_m3,
    'Random Forest': y_pred_m4
}

res_house = []
for name, preds in models_house.items():
    res_house.append({
        'Model': name,
        'MAE (Tỷ VNĐ)': mean_absolute_error(y_test, preds),
        'RMSE (Tỷ VNĐ)': np.sqrt(mean_squared_error(y_test, preds)),
        'R² Score': r2_score(y_test, preds)
    })

res_house_df = pd.DataFrame(res_house)
res_house_df


## 15. Thí Nghiệm 1: So Sánh Mô Hình (Experiment 1: Model Comparison)

Trực quan hóa so sánh hệ số xác định $R^2$ và sai số MAE.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

res_house_df.plot(x='Model', y='R² Score', kind='bar', color='#27ae60', ax=axes[0])
axes[0].set_title('So Sánh Hệ Số Xác Định R² Giữa Các Mô Hình', fontweight='bold')
axes[0].set_ylabel('R² Score (Càng cao càng tốt)')
axes[0].tick_params(axis='x', rotation=30)

res_house_df.plot(x='Model', y='MAE (Tỷ VNĐ)', kind='bar', color='#e67e22', ax=axes[1])
axes[1].set_title('So Sánh Sai Số Tuyệt Đối Trung Bình (MAE)', fontweight='bold')
axes[1].set_ylabel('Tỷ VNĐ (Càng thấp càng tốt)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## 16. Thí Nghiệm 2: Khảo Sát Siêu Tham Số (Experiment 2: Hyperparameter Investigation)

**Câu hỏi thực nghiệm:** Số lượng cây `n_estimators` trong Random Forest ảnh hưởng như thế nào đến hệ số $R^2$?

In [ ]:
n_trees_list = [20, 50, 100, 150, 200, 300]
r2_scores = []

for n in n_trees_list:
    rf_exp = RandomForestRegressor(n_estimators=n, max_depth=10, random_state=42, n_jobs=-1)
    rf_exp.fit(X_train, y_train)
    r2_scores.append(r2_score(y_test, rf_exp.predict(X_test)))

plt.figure(figsize=(8, 4.5))
plt.plot(n_trees_list, r2_scores, 'o-', color='#16a085', linewidth=2)
plt.title('Khảo Sát Siêu Tham Số: Số Lượng Cây vs R² Score', fontweight='bold')
plt.xlabel('Số lượng cây quyết định (n_estimators)')
plt.ylabel('Hệ số R² trên tập Test')
plt.grid(True, alpha=0.3)
plt.show()


## 17. Thí Nghiệm 3: Khảo Sát Biểu Diễn Đặc Trưng (Experiment 3: Feature Investigation)

Đánh giá mức độ đóng góp (Feature Importance) của các đặc trưng trong bài toán định giá BĐS.

In [ ]:
importances = m4.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='Greens_r')
plt.title('Tầm Quan Trọng Của Các Đặc Trưng Trong Định Giá BĐS', fontweight='bold')
plt.xlabel('Feature Importance Score')
plt.show()


## 18. Mô Hình Cuối Cùng & Đóng Gói (Final Model Selection & Persistence)

Lưu trữ mô hình Random Forest Regressor tối ưu và metadata bộ mã hóa vào `.pkl`.

In [ ]:
joblib.dump(m4, 'best_model.pkl')
joblib.dump({'encoders': encoders, 'features': feature_cols}, 'model_metadata.pkl')
print("Đã lưu thành công best_model.pkl và model_metadata.pkl")


## 19. Ứng Dụng Định Giá (Application Pipeline Implementation)

Hàm định giá tiếp nhận thông số căn nhà từ người dùng và tính toán giá trị thị trường.

In [ ]:
def predict_house_price(input_dict):
    loaded_model = joblib.load('best_model.pkl')
    loaded_meta = joblib.load('model_metadata.pkl')
    
    row_df = pd.DataFrame([input_dict])
    for col, le in loaded_meta['encoders'].items():
        if col in row_df.columns:
            val = str(row_df[col].iloc[0])
            if val in le.classes_:
                row_df[col] = le.transform([val])[0]
            else:
                row_df[col] = 0
                
    pred_price = loaded_model.predict(row_df[loaded_meta['features']])[0]
    area = input_dict.get('Area', 50)
    price_per_m2 = (pred_price * 1000) / area if area > 0 else 0
    
    return {
        'Predicted_Price': f'{pred_price:.2f} Tỷ VNĐ',
        'Price_per_m2': f'{price_per_m2:.1f} Triệu VNĐ / m²',
        'Price_Raw': float(pred_price)
    }

print("Đã khởi tạo hoàn tất hàm định giá predict_house_price().")


## 20. Minh Chứng Hệ Thống (System Demonstration - 3 Test Cases)

Kiểm thử định giá 3 căn nhà thực tế tại các phân khúc thị trường.

In [ ]:
test_houses = [
    {'Name': 'Nhà phố Cầu Giấy (Cao cấp)', 'data': {'City': 'Hà Nội', 'Area': 65.0, 'Frontage': 4.5, 'Access_Road': 4.0, 'Floors': 4, 'Bedrooms': 4, 'Bathrooms': 4, 'Legal': 'Sổ đỏ/ Sổ hồng', 'House_direction': 'Đông - Nam', 'Balcony_direction': 'Đông - Nam', 'Furniture': 'Đầy đủ'}},
    {'Name': 'Nhà ngõ nhỏ Hoàng Mai (Bình dân)', 'data': {'City': 'Hà Nội', 'Area': 32.0, 'Frontage': 3.0, 'Access_Road': 2.0, 'Floors': 3, 'Bedrooms': 2, 'Bathrooms': 2, 'Legal': 'Sổ đỏ/ Sổ hồng', 'House_direction': 'Tây', 'Balcony_direction': 'Tây', 'Furniture': 'Cơ bản'}},
    {'Name': 'Biệt thự Thủ Đức TP.HCM (Siêu sang)', 'data': {'City': 'TP.HCM', 'Area': 180.0, 'Frontage': 10.0, 'Access_Road': 8.0, 'Floors': 3, 'Bedrooms': 5, 'Bathrooms': 5, 'Legal': 'Sổ đỏ/ Sổ hồng', 'House_direction': 'Nam', 'Balcony_direction': 'Nam', 'Furniture': 'Cao cấp'}}
]

for th in test_houses:
    res = predict_house_price(th['data'])
    print("=== " + th["Name"] + " ===")
    print("  Định giá: " + res["Predicted_Price"] + " | Đơn giá: " + res["Price_per_m2"])


## 21. Phản Ánh & Chiêm Nghiệm (Reflection on Intelligence & Representation)

**7 Câu hỏi bản chất thông minh & 8 câu hỏi biểu diễn:**
• Giá trị bất động sản phụ thuộc mạnh vào yếu tố vị trí địa lý và hạ tầng xung quanh.
• Vector đặc trưng nắm bắt được các thông số vật lý định lượng (Diện tích, Mặt tiền) nhưng đánh mất thông tin hình ảnh thực tế (Image/Tensor) và mạng lưới tiện ích lân cận.
• Đồ thị tri thức (Knowledge Graph) là cầu nối hoàn hảo để mô hình hóa mối quan hệ giữa Bất động sản $\leftrightarrow$ Quận/Huyện $\leftrightarrow$ Tiện ích Metro/Trường học.

In [ ]:
print("Reflection: Real Estate Valuation requires both tabular metrics and Knowledge Graph spatial connections.")


## 22. Kết Luận (Conclusion)

Hoàn thành trọn vẹn quy trình xây dựng Hệ thống Thông minh định giá bất động sản với mô hình Random Forest đạt hệ số $R^2 > 0.72$, sẵn sàng tích hợp vào nền tảng Web & Mobile.

In [ ]:
print("HOUSING VALUATION NOTEBOOK EXECUTED SUCCESSFULLY!")
